# 02 - Data Quality Check
Measure data quality on the bronze tables. This notebook only reports — it never modifies data.

In [ ]:
from pyspark.sql.functions import col, count, when, isnan, current_timestamp, lit

In [ ]:
trips = spark.read.table("ola_lakehouse.bronze.trips")
customers = spark.read.table("ola_lakehouse.bronze.customers")
drivers = spark.read.table("ola_lakehouse.bronze.drivers")

In [ ]:
trips.show()

### Null checks on required trip columns

In [ ]:
total_rows = trips.count()
required_cols = ["trip_id", "customer_id", "driver_id", "status", "fare_amount"]

for c in required_cols:
    null_count = trips.filter(col(c).isNull()).count()
    print(f"{c}: {null_count} nulls out of {total_rows} rows ({round(null_count/total_rows*100,2)}%)")

### Duplicate trip_id check

In [ ]:
duplicate_count = trips.count() - trips.dropDuplicates(["trip_id"]).count()
print(f"Duplicate trip_id rows: {duplicate_count}")

### Referential integrity — orphan foreign keys (anti-join)

In [ ]:
orphan_customers = trips.join(customers, on="customer_id", how="left_anti").filter(col("customer_id").isNotNull())
orphan_drivers = trips.join(drivers, on="driver_id", how="left_anti").filter(col("driver_id").isNotNull())

print(f"Trips with unknown customer_id: {orphan_customers.count()}")
print(f"Trips with unknown driver_id: {orphan_drivers.count()}")

orphan_customers.show()
orphan_drivers.show()

### Persist DQ results for trend tracking across runs

In [ ]:
dq_results = [
    ("bronze.trips", "null_check_customer_id", "PASS" if trips.filter(col("customer_id").isNull()).count() == 0 else "FAIL", trips.filter(col("customer_id").isNull()).count()),
    ("bronze.trips", "null_check_driver_id", "PASS" if trips.filter(col("driver_id").isNull()).count() == 0 else "FAIL", trips.filter(col("driver_id").isNull()).count()),
    ("bronze.trips", "duplicate_trip_id", "PASS" if duplicate_count == 0 else "FAIL", duplicate_count),
    ("bronze.trips", "orphan_customer_fk", "PASS" if orphan_customers.count() == 0 else "FAIL", orphan_customers.count()),
    ("bronze.trips", "orphan_driver_fk", "PASS" if orphan_drivers.count() == 0 else "FAIL", orphan_drivers.count()),
]

dq_df = spark.createDataFrame(dq_results, ["table_name", "check_name", "status", "failed_count"])
dq_df = dq_df.withColumn("run_ts", current_timestamp())
dq_df.show(truncate=False)

dq_df.write.format("delta").mode("append").option("mergeSchema","true").saveAsTable("ola_lakehouse.bronze.dq_results")

In [ ]:
%sql
SELECT check_name, status, failed_count, run_ts
FROM ola_lakehouse.bronze.dq_results
WHERE run_ts = (SELECT MAX(run_ts) FROM ola_lakehouse.bronze.dq_results)
ORDER BY status DESC, check_name